In [3]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    precision_score,
    recall_score,
    confusion_matrix
)

model = joblib.load("../artifacts/credit_risk_model.pkl")
threshold = joblib.load("../artifacts/decision_threshold.pkl")

In [4]:
df = pd.read_csv("../data/raw/credit_risk_dataset.csv")

df = df.drop_duplicates().reset_index(drop=True)

df = df[df["person_age"] <= 100].reset_index(drop=True)

invalid_emp = df["person_emp_length"] > df["person_age"]
df.loc[invalid_emp, "person_emp_length"] = np.nan

In [5]:
X = df.drop(
    columns=[
        "loan_status",
        "loan_grade",
        "loan_int_rate"
    ]
)

y = df["loan_status"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
test_prob = model.predict_proba(X_test)[:, 1]

test_pred = (test_prob >= threshold).astype(int)

In [8]:
fairness_df = X_test.copy()

fairness_df["actual"] = y_test.values
fairness_df["predicted"] = test_pred
fairness_df["probability"] = test_prob

In [9]:
fairness_df["age_group"] = pd.cut(
    fairness_df["person_age"],
    bins=[19, 29, 39, 49, 100],
    labels=[
        "20-29",
        "30-39",
        "40-49",
        "50+"
    ]
)

fairness_df["age_group"].value_counts().sort_index()

age_group
20-29    4655
30-39    1446
40-49     309
50+        73
Name: count, dtype: int64

In [10]:
results = []

for group, group_df in fairness_df.groupby("age_group", observed=True):
    y_true = group_df["actual"]
    y_pred = group_df["predicted"]

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()

    results.append({
        "age_group": group,
        "sample_size": len(group_df),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "false_positive_rate": fp / (fp + tn) if (fp + tn) > 0 else 0
    })

fairness_results = pd.DataFrame(results)

fairness_results

,age_group,sample_size,precision,recall,false_positive_rate
0,20-29,4655,0.728898,0.695735,0.075833
1,30-39,1446,0.665414,0.614583,0.076857
2,40-49,309,0.785714,0.733333,0.048193
3,50+,73,0.692308,0.600000,0.068966


In [11]:
precision_gap = (
    fairness_results["precision"].max()
    - fairness_results["precision"].min()
)

recall_gap = (
    fairness_results["recall"].max()
    - fairness_results["recall"].min()
)

fpr_gap = (
    fairness_results["false_positive_rate"].max()
    - fairness_results["false_positive_rate"].min()
)

print("Precision gap:", round(precision_gap, 3))
print("Recall gap:", round(recall_gap, 3))
print("FPR gap:", round(fpr_gap, 3))

Precision gap: 0.12
Recall gap: 0.133
FPR gap: 0.029


In [12]:
group_rates = (
    fairness_df
    .groupby("age_group", observed=True)
    .agg(
        sample_size=("actual", "size"),
        actual_default_rate=("actual", "mean"),
        predicted_risk_rate=("predicted", "mean")
    )
)

group_rates

,sample_size,actual_default_rate,predicted_risk_rate
age_group,,,
20-29,4655,0.226638,0.216327
30-39,1446,0.199170,0.183956
40-49,309,0.194175,0.181230
50+,73,0.205479,0.178082
